# Entropía diferencial y distribuciones continuas

La entropía diferencial extiende la entropía de Shannon a variables continuas.
Este cuaderno calcula entropías diferenciales, verifica que la Gaussiana maximiza
la entropía para varianza fija, y explora la conexión con la capacidad del canal Gaussiano.

**Artículo asociado:** [Entropía diferencial y fuentes continuas](../../02-teoria-informacion/10-entropia-diferencial.md)

In [ ]:
import math
import random

def log2(x):
    return math.log2(x) if x > 0 else 0.0

def ln(x):
    return math.log(x) if x > 0 else 0.0

print('Entorno listo.')

## Ejemplo 1: entropías diferenciales analíticas

Para varias distribuciones conocemos la fórmula cerrada de $h(X) = -\int f\log f$.

In [ ]:
def h_uniforme(a, b):
    """h(Uniforme[a,b]) = log2(b-a) bits."""
    return log2(b - a)

def h_exponencial(lam):
    """h(Exponencial(lam)) = 1 - log2(lam) bits."""
    return 1.0 - log2(lam)

def h_gaussiana(sigma2):
    """h(N(mu, sigma^2)) = 0.5*log2(2*pi*e*sigma^2) bits."""
    return 0.5 * log2(2 * math.pi * math.e * sigma2)

def h_laplace(b):
    """h(Laplace(0,b)) = log2(2*e*b) bits."""
    return log2(2 * math.e * b)

print('Entropías diferenciales (en bits):')
print()
print('Distribución Uniforme[0, a]:')
for a in [0.5, 1.0, 2.0, 4.0, 8.0]:
    h = h_uniforme(0, a)
    print(f'  a={a:5.1f}  h = {h:7.4f} bits  (negativa si a<1: h={h_uniforme(0,0.5):.4f} para a=0.5)')
    break  # solo mostrar uno, el resto en la tabla

print()
print(f'{"Distribución":30}  {"h(X) (bits)":>12}')
print('-' * 46)
distribuciones = [
    ('Uniforme[0, 0.5]',   h_uniforme(0, 0.5)),
    ('Uniforme[0, 1]',     h_uniforme(0, 1)),
    ('Uniforme[0, 2]',     h_uniforme(0, 2)),
    ('Uniforme[0, 8]',     h_uniforme(0, 8)),
    ('Exponencial(λ=0.5)', h_exponencial(0.5)),
    ('Exponencial(λ=1)',   h_exponencial(1)),
    ('Exponencial(λ=2)',   h_exponencial(2)),
    ('Gaussiana(σ²=0.25)', h_gaussiana(0.25)),
    ('Gaussiana(σ²=1)',    h_gaussiana(1)),
    ('Gaussiana(σ²=4)',    h_gaussiana(4)),
    ('Laplace(b=1)',       h_laplace(1)),
    ('Laplace(b=2)',       h_laplace(2)),
]
for nombre, h in distribuciones:
    print(f'{nombre:30}  {h:12.4f}')

print()
print('Nota: la entropía diferencial puede ser negativa (Uniforme[0,0.5], Exponencial(λ=2)).')

## Ejemplo 2: la Gaussiana maximiza la entropía para varianza fija

Para una varianza $\sigma^2$ fija, la distribución Gaussiana tiene la mayor entropía diferencial.
Verificamos esto numéricamente comparando Gaussiana, Laplace, Uniforme y Exponencial con la misma varianza.

In [ ]:
# Para varianza sigma^2, calcular los parámetros de cada distribución
# y su entropía diferencial

def comparar_distribuciones_misma_varianza(sigma2):
    sigma = math.sqrt(sigma2)

    # Gaussiana: varianza = sigma^2
    h_gauss = h_gaussiana(sigma2)

    # Laplace(0, b): varianza = 2b^2 → b = sigma/sqrt(2)
    b_laplace = sigma / math.sqrt(2)
    h_lap = h_laplace(b_laplace)

    # Uniforme[-a, a]: varianza = a^2/3 → a = sqrt(3)*sigma
    a_unif = math.sqrt(3) * sigma
    h_unif = h_uniforme(-a_unif, a_unif)

    # Exponencial(lambda): varianza = 1/lambda^2 → lambda = 1/sigma
    # (desplazada para tener media 0, varianza no cambia)
    lam_exp = 1.0 / sigma
    h_exp = h_exponencial(lam_exp)

    return h_gauss, h_lap, h_unif, h_exp

print(f'Entropías para distribuciones con la misma varianza σ²:')
print(f'{"σ²":>6}  {"Gaussiana":>12}  {"Laplace":>10}  {"Uniforme":>10}  {"Exponencial":>13}')
print('-' * 58)
for sigma2 in [0.5, 1.0, 2.0, 4.0]:
    h_g, h_l, h_u, h_e = comparar_distribuciones_misma_varianza(sigma2)
    print(f'{sigma2:>6.1f}  {h_g:>12.4f}  {h_l:>10.4f}  {h_u:>10.4f}  {h_e:>13.4f}')

print()
print('La Gaussiana tiene siempre la mayor entropía diferencial para varianza fija.')
print('Orden: Gaussiana > Laplace > Uniforme (para estas tres con misma varianza).')

## Ejemplo 3: invarianza de la entropía diferencial bajo traslación y escala

- $h(X + c) = h(X)$ para cualquier constante $c$.
- $h(aX) = h(X) + \log_2|a|$ para cualquier constante $a \neq 0$.

In [ ]:
# Verificar con la Gaussiana
sigma2 = 1.0

print('Verificación de invarianza para X ~ N(0,1):')
print()
print('Traslación h(X+c) = h(X):')
for c in [-10, -1, 0, 1, 10]:
    # h(N(c, sigma^2)) = h(N(0, sigma^2)) — la media no importa
    h_original = h_gaussiana(sigma2)
    h_trasladada = h_gaussiana(sigma2)  # media no afecta
    print(f'  c={c:>4}:  h(X+c) = {h_trasladada:.4f}  (igual a h(X) = {h_original:.4f})')

print()
print('Escala h(aX) = h(X) + log2|a|  (X ~ N(0,1), aX ~ N(0,a^2)):')
h_base = h_gaussiana(sigma2)
for a in [0.25, 0.5, 1, 2, 4]:
    h_escalada = h_gaussiana(a**2 * sigma2)  # aX ~ N(0, a^2*sigma^2)
    ajuste_teorico = h_base + log2(abs(a))
    ok = '✓' if abs(h_escalada - ajuste_teorico) < 1e-10 else '✗'
    print(f'  {ok}  a={a:5.2f}:  h(aX) = {h_escalada:.4f}  = h(X) + log2({a}) = {ajuste_teorico:.4f}')

## Ejemplo 4: capacidad del canal Gaussiano

Para el canal $Y = X + Z$ con $Z \sim \mathcal{N}(0,N)$ y potencia $\mathbb{E}[X^2] \leq P$,
la capacidad es $C = \frac{1}{2}\log_2(1 + P/N)$ bits/uso.

In [ ]:
def capacidad_gaussiana(P, N):
    """C = 0.5 * log2(1 + P/N) bits/uso."""
    return 0.5 * log2(1 + P / N)

def derivacion_capacidad(P, N):
    """Muestra los pasos de la derivación."""
    sigma2_Y = P + N       # Y ~ N(0, P+N) para X ~ N(0,P)
    h_Y = h_gaussiana(sigma2_Y)
    h_Z = h_gaussiana(N)   # h(Y|X) = h(Z)
    I = h_Y - h_Z
    return h_Y, h_Z, I

N = 1.0  # potencia del ruido
print(f'Canal Gaussiano Y = X + Z, Z~N(0,{N}):')
print(f'{"P (potencia señal)":>20}  {"SNR=P/N":>10}  {"h(Y)":>8}  {"h(Z)":>8}  {"C=I(X;Y)":>10}')
print('-' * 65)
for P in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 100.0]:
    h_Y, h_Z, I = derivacion_capacidad(P, N)
    C = capacidad_gaussiana(P, N)
    snr = P / N
    print(f'{P:>20.2f}  {snr:>10.2f}  {h_Y:>8.4f}  {h_Z:>8.4f}  {I:>10.4f}')

print()
print('Observación: cada duplicación de la potencia P añade aproximadamente 0.5 bit de capacidad.')
print('Para SNR grande: C ≈ 0.5*log2(SNR) — crece logarítmicamente.')

## Ejemplo 5: estimación numérica de la entropía diferencial

Para distribuciones sin fórmula cerrada podemos estimar $h(X)$ numéricamente
usando una rejilla fina (cuantización) y la relación $H(X_\Delta) \approx h(X) - \log_2\Delta$.

In [ ]:
def estimar_h_diferencial(samples, delta):
    """
    Estima h(X) ≈ H(X_delta) + log2(delta) usando cuantización con paso delta.
    """
    # Cuantizar
    bins = {}
    for x in samples:
        k = int(x / delta)
        bins[k] = bins.get(k, 0) + 1
    n = len(samples)
    # Entropía discreta de la versión cuantizada
    H_delta = -sum((c/n) * log2(c/n) for c in bins.values())
    # Corrección
    return H_delta + log2(delta)

def generar_gaussiana(mu, sigma, n, seed=42):
    rng = random.Random(seed)
    return [rng.gauss(mu, sigma) for _ in range(n)]

sigma2 = 1.0
sigma  = math.sqrt(sigma2)
h_teorica = h_gaussiana(sigma2)

print(f'Estimación numérica de h(N(0,1)) (teórico = {h_teorica:.4f} bits):')
print(f'{"n":>10}  {"Δ":>8}  {"Estimado":>12}  {"Error":>10}')
samples_large = generar_gaussiana(0, sigma, 200000)
for n, delta in [(1000, 0.1), (5000, 0.05), (20000, 0.02), (100000, 0.01), (200000, 0.005)]:
    h_est = estimar_h_diferencial(samples_large[:n], delta)
    print(f'{n:>10}  {delta:>8.3f}  {h_est:>12.4f}  {abs(h_est - h_teorica):>10.4f}')

## Ideas clave

- $h(X) = -\int f\log f$ puede ser negativa: no mide bits directamente, sino incertidumbre relativa.
- La Gaussiana maximiza $h(X)$ entre todas las distribuciones con varianza fija: es la distribución
  de máxima incertidumbre bajo restricción de potencia.
- $h(aX) = h(X) + \log_2|a|$; la traslación no cambia la entropía.
- La capacidad del canal Gaussiano $C = \frac{1}{2}\log(1+P/N)$ se obtiene directamente usando
  que $h(Y) - h(Z) = \frac{1}{2}\log(P+N) - \frac{1}{2}\log N$.
- La estimación numérica requiere cuantización fina y muchas muestras; el error disminuye lentamente.